# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SWAPI03/flyrank-ai-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I read *The State of AI-Driven SEO* (FlyRank, March 2026) the way the session read it: constructively.
The paper discloses its standards up front (observational, minimum 50 per bucket, ML kept to an
exploratory appendix), so these are questions I'd want asked of my own work, not grades.

**Finding #1 — "The Anatomy of Growing Content" (CONFIRMED).** Rising pages are longer (3.2K vs 2.3K
words), younger (184 vs 230 days), and slightly better positioned than declining pages.
*My methodology question — where does the label come from?* The up/down split is defined from the
30-day-vs-previous-30-day impression change, a bucket derived from the same impression series it then
describes, and the comparison is cross-sectional. So it cleanly supports the **observed association**
"rising pages tend to be longer," but the action it motivates ("expand thin pages so they keep
growing") is a **causal** claim the design can't carry — length may be a proxy for topic maturity.
The paper already labels this observational, which is the right disclosure; I'd only ask that the
recommendation be framed as directional rather than expected-to-cause.

**Finding #2 — "The Content Performance Curve" (CONFIRMED).** Health score peaks at 61-90 days,
decays at 271-365, and rebounds at 365+.
*My methodology question — does the validation design support the claim?* This is a cross-sectional
comparison of **different** pages at different ages, not one cohort followed over time, so it
confounds age with vintage and survivorship: the 365+ "rebound" is explicitly concentrated in pages
that were refreshed (a selection effect the paper discloses). And `health_score` is a composite that
already contains position and CTR, so "performance by age" partly measures the scoring formula. To
turn this into a within-page *lifecycle* claim I'd want a longitudinal cohort tracked across months.
Again the paper's own caveat about the refreshed 365+ pages is exactly the right move.

In [1]:
# Setup + load + rebuild the Week-5 review population and model, so I can re-audit it here.
import os, sys, json, subprocess
import numpy as np, pandas as pd, sklearn
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
d = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
       (df["avg_position"] <= 20) & (df["ctr"] > 0)].copy()
d["log_impressions"] = np.log1p(d["impressions_90d"])
NUM = ["avg_position","log_impressions","content_age_days","days_since_last_update",
       "word_count","search_volume","competition","cpc"]
CAT = ["content_type","main_intent"]
for c in NUM: d[c] = pd.to_numeric(d[c], errors="coerce")
d[NUM] = d[NUM].fillna(d[NUM].median()); d[CAT] = d[CAT].fillna("unknown")
d["ctr_c"] = d["ctr"].clip(upper=d["ctr"].quantile(0.99))
pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough")

def fit_eval(train_idx, test_idx, num=NUM):
    m = Pipeline([("pre", ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), CAT)],
                                            remainder="passthrough")),
                  ("m", GradientBoostingRegressor(random_state=42))])
    m.fit(d.iloc[train_idx][num + CAT], d.iloc[train_idx]["ctr_c"])
    pred = m.predict(d.iloc[test_idx][num + CAT])
    return mean_absolute_error(d.iloc[test_idx]["ctr_c"], pred), r2_score(d.iloc[test_idx]["ctr_c"], pred)

print("scikit-learn", sklearn.__version__)
print(f"review population: {len(d):,} pages across {d['client_id'].nunique()} clients")

scikit-learn 1.7.2
review population: 10,807 pages across 28 clients


## 2. My model under an honest split (before/after)

The Week-5 model already used a client-grouped split — here I show *why* by putting it next to the
naive random split. Same model, same data, same metric; only the split changes. The gap between the
two numbers is itself the finding: how much "skill" was really memorization of clients seen in
training.

In [2]:
idx = np.arange(len(d))

# BEFORE: naive random split (rows from one client can land in both train and test).
tr_r, te_r = train_test_split(idx, test_size=0.25, random_state=42)
mae_r, r2_r = fit_eval(tr_r, te_r)

# AFTER: honest client-grouped split (every test client is unseen in training).
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_g, te_g = next(gss.split(d, d["ctr_c"], d["client_id"]))
mae_g, r2_g = fit_eval(tr_g, te_g)

comp = pd.DataFrame({
    "split": ["random (before)", "client-grouped (after)"],
    "MAE":   [round(mae_r, 4), round(mae_g, 4)],
    "R2":    [round(r2_r, 4), round(r2_g, 4)],
    "test_rows": [len(te_r), len(te_g)],
}).set_index("split")
print(comp.to_string())
print(f"\nObserved: moving to an honest client split, R2 falls from {r2_r:.3f} to {r2_g:.3f} and MAE")
print("rises. That drop is memorization the random split was hiding. The honest reading is that on")
print("UNSEEN clients the model's absolute-CTR accuracy is only directional -- decision-support, not")
print("a precise predictor. Every number I report from here uses the grouped split.")

                           MAE      R2  test_rows
split                                            
random (before)         0.2111  0.1429       2702
client-grouped (after)  0.2864  0.0013        724

Observed: moving to an honest client split, R2 falls from 0.143 to 0.001 and MAE
rises. That drop is memorization the random split was hiding. The honest reading is that on
UNSEEN clients the model's absolute-CTR accuracy is only directional -- decision-support, not
a precise predictor. Every number I report from here uses the grouped split.


## 3. Leakage audit

The Week-3 hunt, run on my final feature set. Three checks: (1) enumerate features and confirm none
are label-derived or product flags, (2) draw the timeline, (3) the confession test — add a suspect
column and watch the score jump.

In [3]:
# (1) Feature inventory vs banned columns.
banned = ["ctr", "clicks_90d", "clicks_last_30d", "trend_direction", "trend_pct",
          "is_declining_label", "engagement_rate", "health_score", "priority_score"]
in_features = [c for c in NUM + CAT if c in banned]
print("Final features:", NUM + CAT)
print("Banned (label-derived / future / product-flag) columns used as features:",
      in_features if in_features else "none")
assert not in_features, "leakage: a banned column is in the feature set"

# (2) Timeline / population honesty note.
print("\nTimeline: all features are 90-day-window observables known at scoring time; the target is")
print("same-window CTR (a scoring proxy, not a future label). Limitation stated: feature and target")
print("share the 90-day window, so this is opportunity SCORING, not future prediction -- the warehouse")
print("past->future label is the upgrade.")

# (3) Confession test: add clicks_90d (clicks + impressions reconstruct CTR) and watch R2 spike.
d["clicks_90d_f"] = pd.to_numeric(d["clicks_90d"], errors="coerce").fillna(0)
mae_clean, r2_clean = fit_eval(tr_g, te_g, num=NUM)
mae_leak,  r2_leak  = fit_eval(tr_g, te_g, num=NUM + ["clicks_90d_f"])
print(f"\nConfession test (client-grouped):")
print(f"  clean features        R2 {r2_clean:.3f}  MAE {mae_clean:.3f}")
print(f"  + clicks_90d (LEAK)   R2 {r2_leak:.3f}  MAE {mae_leak:.3f}   <- near-perfect = the answer in disguise")
print("A jump this large is the signature of leakage: clicks/impressions IS ctr. clicks stays OUT.")

Final features: ['avg_position', 'log_impressions', 'content_age_days', 'days_since_last_update', 'word_count', 'search_volume', 'competition', 'cpc', 'content_type', 'main_intent']
Banned (label-derived / future / product-flag) columns used as features: none

Timeline: all features are 90-day-window observables known at scoring time; the target is
same-window CTR (a scoring proxy, not a future label). Limitation stated: feature and target
share the 90-day window, so this is opportunity SCORING, not future prediction -- the warehouse
past->future label is the upgrade.



Confession test (client-grouped):
  clean features        R2 0.001  MAE 0.286
  + clicks_90d (LEAK)   R2 0.978  MAE 0.031   <- near-perfect = the answer in disguise
A jump this large is the signature of leakage: clicks/impressions IS ctr. clicks stays OUT.


## 4. Claim rewrite

**My boldest earlier sentence (overreaching):**
> "My model finds the pages that will gain clicks once their titles are fixed."

**Rewritten in safe language:**
> On held-out clients, the model produces a position-adjusted *expected CTR*; ranking pages by the
> gap between expected and actual CTR **surfaces pages observed to under-capture clicks relative to
> peers at the same position**. This is a **decision-support** ranking for human review. It does
> **not** establish that editing a title *causes* more clicks — that needs an experiment — and under
> the honest client-grouped split the model's absolute-CTR accuracy is only **directional**, barely
> ahead of a simple tier median on unseen clients.

**And my Week-1 finding, tightened the same way:**
> Bold: "Comparison articles have the worst CTR." →
> Safe: "In this sample, comparison articles were **observed** to have the lowest mean CTR across
> most position tiers — a **directional, observational** pattern, not a claim about Google's ranking
> or about causation."

The attack-checklist receipts print below.

In [4]:
print("Attack checklist (this run):")
print(f"  [x] honest split reported next to random  -> R2 {r2_g:.3f} (grouped) vs {r2_r:.3f} (random)")
print(f"  [x] base/naive reference stated           -> tier-median baseline from Week 5")
print(f"  [x] no label-derived/future/flag features -> banned-in-features: none")
print(f"  [x] leakage confession test run           -> +clicks_90d R2 {r2_leak:.3f} (rejected)")
print(f"  [x] population limitation disclosed        -> feature & target share the 90-day window")
print("All claims kept to: observed, measured, directional, decision-support.")

Attack checklist (this run):
  [x] honest split reported next to random  -> R2 0.001 (grouped) vs 0.143 (random)
  [x] base/naive reference stated           -> tier-median baseline from Week 5
  [x] no label-derived/future/flag features -> banned-in-features: none
  [x] leakage confession test run           -> +clicks_90d R2 0.978 (rejected)
  [x] population limitation disclosed        -> feature & target share the 90-day window
All claims kept to: observed, measured, directional, decision-support.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.